In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
import json
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    """实验超参数配置"""
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数
        self.embed_dim = 64
        self.seq_len = 100
        self.tcan_layers = 2
        self.kernel_size = 3
        self.dropout = 0.3

        # 训练参数
        self.batch_size = 64
        self.epochs = 10
        self.lr = 0.001
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径
        self.data_dir = "./data"
        self.dataset_name = "math2015"
        self.fallback_url = ""

# ==========================================
# 2. 数据处理模块 (矩阵适配版 Data Processor)
# ==========================================

class AssistDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx]

        seq_len = len(q_seq)

        if seq_len >= self.max_len:
            q_seq = q_seq[-self.max_len:]
            r_seq = r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long)
        )

class DataProcessor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _install_edudata(self):
        print("正在尝试自动安装 EduData 库...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
            print("EduData 安装成功！")
            return True
        except Exception as e:
            print(f"自动安装失败: {e}")
            return False

    def download_data(self):
        existing_file = self._find_data_file(self.config.data_dir)
        if existing_file:
            print(f"发现已存在的数据文件: {existing_file}")
            return existing_file

        print(f"正在准备下载数据集: {self.config.dataset_name}")
        try:
            from EduData import get_data
            get_data(self.config.dataset_name, self.config.data_dir)
        except ImportError:
            if self._install_edudata():
                try:
                    from EduData import get_data
                    get_data(self.config.dataset_name, self.config.data_dir)
                except Exception as e:
                    print(f"下载失败: {e}")
            else:
                print("无法安装 EduData，请手动下载数据集。")
        except Exception as e:
            print(f"EduData 下载过程出错: {e}")

        return self._find_data_file(self.config.data_dir)

    def _find_data_file(self, root_dir):
        valid_extensions = ['.csv', '.txt', '.tsv', '.dat', '.json']
        candidates = []
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.startswith('.'): continue
                if any(file.lower().endswith(ext) for ext in valid_extensions):
                    # 排除 q.txt (Q-matrix)
                    if 'q.txt' in file.lower(): continue
                    full_path = os.path.join(root, file)
                    candidates.append(full_path)

        if not candidates: return None
        # 优先 data.txt
        candidates.sort(key=lambda x: (not ('data' in os.path.basename(x).lower()), -os.path.getsize(x)))
        return candidates[0]

    def process(self):
        file_path = self.download_data()
        if not file_path:
            raise FileNotFoundError("未找到有效数据文件。")

        print(f"读取数据: {file_path} ...")

        # 1. 尝试读取
        try:
            # 尝试自动推断
            df = pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8')
        except:
            df = pd.read_csv(file_path, sep=None, engine='python', encoding='latin-1')

        # 2. 列名映射
        cols = [str(c).lower() for c in df.columns]
        col_map = {str(c).lower(): c for c in df.columns}

        user_col = next((col_map[c] for c in cols if 'user' in c or 'student' in c or 'uid' in c), None)
        item_col = next((col_map[c] for c in cols if 'problem' in c or 'item' in c or 'pid' in c), None)
        skill_col = next((col_map[c] for c in cols if 'skill' in c or 'concept' in c or 'kc' in c), None)
        correct_col = next((col_map[c] for c in cols if 'correct' in c or 'score' in c or 'answer' in c), None)

        # 3. 关键修复: 矩阵格式处理
        # 如果找不到 user/item 列，且列数很多，或者是纯数字列名，则假设是 Matrix 格式
        is_matrix = False
        if (user_col is None or item_col is None):
            # 检查是否为宽表 (列数 > 5)
            if df.shape[1] > 5:
                print(">>> 检测到 User-Item 响应矩阵格式 (无表头)，正在转换...")
                is_matrix = True

                # 重新读取，不带表头
                try:
                    df = pd.read_csv(file_path, sep=None, engine='python', header=None)
                except:
                    pass # 使用当前df

                # 假设行号=User, 列号=Item
                df.index.name = 'user_id'
                df.columns.name = 'item_id'

                # 转换为长表: User, Item, Score
                df = df.reset_index().melt(id_vars='user_id', var_name='item_id', value_name='correct')

                # 更新列名
                user_col = 'user_id'
                item_col = 'item_id'
                correct_col = 'correct'

                # 清洗无效值
                df.dropna(subset=[correct_col], inplace=True)
                print(f"转换完成，样本数: {len(df)}")

        if not (user_col and item_col and correct_col):
            print("数据前5行预览:")
            print(df.head())
            raise ValueError("关键列缺失且无法自动识别为矩阵格式。")

        # 4. 寻找 Skill 信息 (Q-Matrix)
        if not skill_col:
            # 尝试寻找同目录下的 q.txt
            data_dir = os.path.dirname(file_path)
            q_file = os.path.join(data_dir, 'q.txt')
            if not os.path.exists(q_file):
                # 尝试其他可能的 Q 矩阵文件
                for f in os.listdir(data_dir):
                    if 'q.txt' in f.lower() or 'q_matrix' in f.lower():
                        q_file = os.path.join(data_dir, f)
                        break

            if os.path.exists(q_file):
                print(f"发现 Q-Matrix 文件: {q_file}，正在加载...")
                try:
                    # 假设 Q-Matrix 也是矩阵格式: Item x Skill
                    q_df = pd.read_csv(q_file, sep=None, engine='python', header=None)
                    # 找到每个 Item 对应的 Skill (假设每一行是一个 Item，列是 Skill，值为 1)
                    # 转换为 Item -> Skill 映射
                    item_skill_map = {}
                    for idx, row in q_df.iterrows():
                        # 找到值为 1 的列索引作为 Skill ID
                        skills = row[row == 1].index.tolist()
                        if skills:
                            item_skill_map[idx] = skills[0] # 简化：取第一个 Skill
                        else:
                            item_skill_map[idx] = 0 # 无 Skill

                    # 映射回主数据
                    # 注意：如果 df['item_id'] 是从 0 开始的索引，可以直接 map
                    # 这里假设 item_id 对应行号
                    skill_col = 'skill_id_mapped'
                    df[skill_col] = df[item_col].map(item_skill_map).fillna(0)
                    print("Q-Matrix 映射成功。")
                except Exception as e:
                    print(f"加载 Q-Matrix 失败 ({e})，回退到 Item=Skill 模式。")
                    skill_col = 'temp_skill_id'
                    df[skill_col] = df[item_col]
            else:
                print("警告: 未找到 Skill 列且未发现 q.txt。将使用 Item ID 代替 Skill ID。")
                skill_col = 'temp_skill_id'
                df[skill_col] = df[item_col]

        # 数据清洗
        df.dropna(subset=[skill_col], inplace=True)
        df[correct_col] = pd.to_numeric(df[correct_col], errors='coerce')
        df.dropna(subset=[correct_col], inplace=True)
        df[correct_col] = df[correct_col].apply(lambda x: 1 if x >= 0.5 else 0)

        # ID 映射
        def get_map(unique_vals):
            return {val: i+1 for i, val in enumerate(unique_vals)}

        user_map = get_map(df[user_col].unique())
        df['user_id_mapped'] = df[user_col].map(user_map)
        self.config.num_students = len(user_map) + 1

        prob_map = get_map(df[item_col].unique())
        df['item_id'] = df[item_col].map(prob_map)
        self.config.num_questions = len(prob_map) + 1

        skill_map = get_map(df[skill_col].unique())
        df['skill_id'] = df[skill_col].map(skill_map)
        self.config.num_concepts = len(skill_map) + 1

        # 计算难度
        print("计算题目难度...")
        item_difficulty = df.groupby('item_id')[correct_col].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, diff in item_difficulty.items():
            diff_values[iid] = diff

        print(f"统计: 学生 {self.config.num_students}, 题目 {self.config.num_questions}, 知识点 {self.config.num_concepts}")

        # 构建 Q-Matrix
        print("构建归一化 Q-Matrix...")
        q_k_relation = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp_df = df[['item_id', 'skill_id']].drop_duplicates()
        for _, row in tmp_df.iterrows():
            q_k_relation[int(row['item_id']), int(row['skill_id'])] = 1

        row_sums = q_k_relation.sum(axis=1)
        row_sums[row_sums == 0] = 1
        q_k_relation = q_k_relation / row_sums[:, np.newaxis]

        # 生成序列
        print("生成序列...")
        all_q, all_r, all_s = [], [], []

        # 完整数据运行
        df_small = df

        for uid, group in tqdm(df_small.groupby('user_id_mapped')):
            if len(group) < 5: continue
            # 如果是矩阵转换来的，可能没有时间顺序，按 Item ID 排序或保持原样
            # 这里简单处理：不排序，或者按 Item ID 排序
            # group = group.sort_values('item_id')
            all_q.append(group['item_id'].values)
            all_r.append(group[correct_col].values)
            all_s.append(uid)

        return all_q, all_r, all_s, q_k_relation, diff_values

# ==========================================
# 3. 核心模型 (Optimized Model)
# ==========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, diff_values):
        super(HeteroGraphEmbedding, self).__init__()
        self.embed_dim = config.embed_dim
        self.question_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.concept_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)
        self.register_buffer('diff_values', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Linear(1, config.embed_dim)

    def forward(self, question_ids, q_k_relation):
        k_emb_weight = self.concept_emb.weight
        q_k_agg = torch.matmul(q_k_relation, k_emb_weight)
        d_val = self.diff_values[question_ids].unsqueeze(-1)
        d_feat = self.diff_proj(d_val)
        q_id_feat = self.question_emb(question_ids)
        q_k_feat = F.embedding(question_ids, q_k_agg, padding_idx=0)
        return q_id_feat + q_k_feat + d_feat

class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super(TemporalAttention, self).__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, L, D = x.size()
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(self.dropout(attn), V)

class TCANBlock(nn.Module):
    def __init__(self, embed_dim, kernel_size, dilation, dropout):
        super(TCANBlock, self).__init__()
        self.ta = TemporalAttention(embed_dim, dropout)
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, dilation=dilation, padding=self.padding)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        z = self.ta(x)
        z_perm = z.permute(0, 2, 1)
        conv_out = self.conv(z_perm)
        conv_out = conv_out[:, :, :z.size(1)].permute(0, 2, 1)
        out = self.norm(conv_out + residual)
        return self.dropout(self.relu(out))

class HIG_TCAN_CD_Optimized(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super(HIG_TCAN_CD_Optimized, self).__init__()
        self.config = config
        self.register_buffer('q_k_relation', torch.tensor(q_k_relation, dtype=torch.float32))
        self.graph_emb = HeteroGraphEmbedding(config, diff_values)
        self.student_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.tcan_layers = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])
        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q_ids, answers, student_ids):
        q_emb = self.graph_emb(q_ids, self.q_k_relation)
        x = torch.cat([q_emb, answers.unsqueeze(-1)], dim=-1)
        x = self.input_proj(x)
        h = x
        for layer in self.tcan_layers:
            h = layer(h)
        s_static = self.student_emb(student_ids).unsqueeze(1).expand(-1, q_ids.size(1), -1)
        return h, q_emb, s_static

# ==========================================
# 4. 实验流程 (Execution)
# ==========================================

def evaluate(model, loader, config):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for q, r, mask, s in loader:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
            h_seq, q_seq_emb, s_static = model(q, r, s)
            h_pred = h_seq[:, :-1, :]
            q_next = q_seq_emb[:, 1:, :]
            s_static_seq = s_static[:, 1:, :]
            feat = torch.cat([h_pred, q_next, s_static_seq], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)
            valid_mask = mask[:, 1:].bool()
            if valid_mask.sum() == 0: continue
            flat_preds = preds[valid_mask].cpu().numpy()
            flat_targets = r[:, 1:][valid_mask].cpu().numpy()
            all_preds.append(flat_preds)
            all_targets.append(flat_targets)
    if len(all_preds) == 0: return
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    acc = accuracy_score(all_targets, (all_preds >= 0.5).astype(int))
    try: auc = roc_auc_score(all_targets, all_preds)
    except: auc = 0.0
    print(f"Test ACC: {acc:.4f} | Test AUC: {auc:.4f}")

def run_experiment():
    print(">>> 阶段 1: 数据准备 (Math2015)")
    config = Config()
    processor = DataProcessor(config)
    try:
        q_seqs, r_seqs, s_seqs, q_k_rel, diff_values = processor.process()
    except Exception as e:
        print(f"数据处理错误: {e}")
        return
    if len(q_seqs) == 0: return
    train_q, test_q, train_r, test_r, train_s, test_s = train_test_split(q_seqs, r_seqs, s_seqs, test_size=0.2, random_state=42)
    train_dataset = AssistDataset(train_q, train_r, train_s, config.seq_len)
    test_dataset = AssistDataset(test_q, test_r, test_s, config.seq_len)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
    print(f"\n>>> 阶段 2: 模型初始化 (Device: {config.device})")
    model = HIG_TCAN_CD_Optimized(config, q_k_rel, diff_values).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    print("\n>>> 阶段 3: 开始训练")
    for epoch in range(config.epochs):
        model.train()
        total_loss = 0
        steps = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs}")
        for q, r, mask, s in pbar:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
            optimizer.zero_grad()
            h_seq, q_seq_emb, s_static = model(q, r, s)
            h_pred = h_seq[:, :-1, :]
            q_next = q_seq_emb[:, 1:, :]
            s_static_seq = s_static[:, 1:, :]
            r_target = r[:, 1:]
            mask_target = mask[:, 1:]
            feat = torch.cat([h_pred, q_next, s_static_seq], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)
            loss = F.binary_cross_entropy(preds, r_target, reduction='none')
            masked_loss = (loss * mask_target).sum() / (mask_target.sum() + 1e-8)
            masked_loss.backward()
            optimizer.step()
            total_loss += masked_loss.item()
            steps += 1
            pbar.set_postfix({'Loss': f"{masked_loss.item():.4f}"})
        scheduler.step()
        print(f"Epoch {epoch+1} Avg Loss: {total_loss/steps:.4f}")
        evaluate(model, test_loader, config)

if __name__ == "__main__":
    run_experiment()

>>> 阶段 1: 数据准备 (Math2015)
正在准备下载数据集: math2015
正在尝试自动安装 EduData 库...
EduData 安装成功！


downloader, INFO http://staff.ustc.edu.cn/~qiliuql/data/math2015.rar is saved as data/math2015.rar


downloader, INFO data/math2015.rar is unrar to data/math2015



读取数据: ./data/math2015/math2015/Math1/data.txt ...
>>> 检测到 User-Item 响应矩阵格式 (无表头)，正在转换...
转换完成，样本数: 84180
发现 Q-Matrix 文件: ./data/math2015/math2015/Math1/q.txt，正在加载...
Q-Matrix 映射成功。
计算题目难度...
统计: 学生 4210, 题目 21, 知识点 7
构建归一化 Q-Matrix...
生成序列...


100%|██████████| 4209/4209 [00:00<00:00, 9368.12it/s]


...

Epoch 3/10: 100%|██████████| 53/53 [00:11<00:00,  4.65it/s, Loss=0.5413]


Epoch 3 Avg Loss: 0.5382
Test ACC: 0.7138 | Test AUC: 0.7913


Epoch 4/10: 100%|██████████| 53/53 [00:10<00:00,  4.98it/s, Loss=0.5322]


Epoch 4 Avg Loss: 0.5359
Test ACC: 0.7156 | Test AUC: 0.7918


Epoch 5/10:  79%|███████▉  | 42/53 [00:08<00:02,  5.20it/s, Loss=0.5159]